# 1. Treatment timing

Applied Econometrics project — *Sanctions and trade: staggered adoption, terminations, the extensive margin, and diversion*.

This notebook takes the merged GSDB-R4 x CEPII panel built for the ML-Econ project and constructs the treatment-timing objects that every downstream estimator needs.

**What it does**

| Step | Purpose |
|---|---|
| (a) | Panel integrity checks |
| (b) | Recover pre-sample sanction status from raw GSDB — the left-censoring fix |
| (c) | Build sanction spells, onset and termination events |
| (d) | Classify each dyad into an estimation group |
| (e) | Transitions table — the feasibility gate for the project |

**Why left-censoring matters.** The panel starts in 1995 but GSDB starts in 1950. A dyad already sanctioned in 1995 (US–Cuba, US–PRK) shows `sanctioned_any = 1` from its first observed year. Assigning it cohort `G = 1995` would make a pre-existing *level* difference look like an *event*. Those dyads are excluded from the onset analysis and kept for the termination analysis, where their spell end is genuinely observable.

**Direction.** Built on `merged_main_final.csv` (sender → exporter). The robustness panel has the same timing structure; re-run with `PANEL_FILE` swapped to confirm.

**Outputs:** `panel_timing.csv`, `dyad_classification.csv`, `transitions.csv`.

#### Libraries

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- paths ---
CLEAN_PATH = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
SANC_PATH  = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Initial/gsdb_v4/'
OUT_PATH   = '/content/drive/MyDrive/IMT_studies/Applied_econ/Project/Data/'

os.makedirs(OUT_PATH, exist_ok=True)

# --- constants ---
PANEL_FILE  = "merged_main_final.csv"   # swap to merged_robust_final.csv for the robustness run
PANEL_START = 1995                       # first year of BACI trade coverage
PANEL_END   = 2020

## (a) Load and verify

Cheap assertions that would invalidate everything downstream if they failed. The pre-1995 check is the important one: BACI has no coverage before 1995, so any surviving pre-1995 row would carry a fabricated `trade = 0` from the zero-fill step.

In [ ]:
%%time
df = pd.read_csv(CLEAN_PATH + PANEL_FILE,
                 dtype={"exp_iso3": str, "imp_iso3": str})

print("shape:", df.shape)
print("year range:", df["year"].min(), "-", df["year"].max())

shape: (1404170, 54)
year range: 1995 - 2020
CPU times: user 12.4 s, sys: 3.58 s, total: 16 s
Wall time: 22.1 s


In [ ]:
# --- integrity checks ---

# 1. no pre-BACI rows (their trade==0 would be a fill artifact)
n_pre = int((df["year"] < PANEL_START).sum())
assert n_pre == 0, f"{n_pre} rows before {PANEL_START}: trade==0 there is fabricated"
print(f"pre-{PANEL_START} rows: 0                [OK]")

# 2. dyad-year uniqueness (needed for lag operations below)
n_dup = int(df.duplicated(["exp_iso3", "imp_iso3", "year"]).sum())
assert n_dup == 0, f"{n_dup} duplicate dyad-years"
print("duplicate dyad-years: 0          [OK]")

# 3. zero-fill completed
assert df["trade"].isna().sum() == 0, "NaN trade remains"
print("NaN trade: 0                     [OK]")

# 4. zero share
print(f"share zero trade: {(df['trade'] == 0).mean():.3f}")

pre-1995 rows: 0                [OK]
duplicate dyad-years: 0          [OK]
NaN trade: 0                     [OK]
share zero trade: 0.509


#### Pair identifier and observation windows

The panel is **unbalanced by construction**: existence gating dropped dyad-years in which a country did not yet or no longer existed (SSD before 2011, and so on). Onsets and terminations must therefore be defined relative to each dyad's own observed window, not to 1995–2020 globally. A `0 -> 1` change across a *gap* year is not an event.

In [ ]:
df["pair"] = df["exp_iso3"] + "_" + df["imp_iso3"]
print(f"directed dyads: {df['pair'].nunique():,}")

span = df.groupby("pair")["year"].agg(first_obs="min", last_obs="max", n_obs="count")
full_len = PANEL_END - PANEL_START + 1
print(f"dyads observed all {full_len} years: {int((span['n_obs'] == full_len).sum()):,} "
      f"of {len(span):,}")
span["n_obs"].value_counts().sort_index(ascending=False).head(10)

directed dyads: 55,922
dyads observed all 26 years: 51,756 of 55,922


,count
n_obs,
26,51756
19,914
16,456
15,922
12,458
11,464
10,934
9,4
5,8


In [ ]:
# dyads with a hole in the MIDDLE of their observed window,
# as opposed to a late start or early end
def has_interior_gap(years):
    ys = sorted(years)
    return (ys[-1] - ys[0] + 1) != len(ys)

interior = df.groupby("pair")["year"].apply(lambda s: has_interior_gap(s.values))
print("dyads with an interior gap:", int(interior.sum()))

dyads with an interior gap: 0


the panel is unbalanced only through country entry and exit, never through interior missingness, so onset and termination timing is exactly identified wherever a dyad is observed in consecutive years.

## (b) Left-censoring: sanction status before 1995

Merge back to raw `GSDB_V4_dyadic.dta` to see which dyads were already under sanction when the panel opens. Apply the same two cleaning steps as notebook 1 of the ML project (BYS→BLR recode, drop non-state targets) so the codes line up.

In [ ]:
%%time
gsdb = pd.read_stata(SANC_PATH + "GSDB_V4_dyadic.dta")

# same cleaning as ML project notebook 1, so ISO codes match the panel
for c in ["sanctioning_state_iso3", "sanctioned_state_iso3"]:
    gsdb[c] = gsdb[c].replace({"BYS": "BLR"})
gsdb = gsdb[gsdb["sanctioned_state_iso3"] != ""].copy()   # drop terrorist-org targets

if pd.api.types.is_datetime64_any_dtype(gsdb["year"]):
    gsdb["year"] = gsdb["year"].dt.year
gsdb["year"] = gsdb["year"].astype("int64")

print("GSDB shape:", gsdb.shape)
print("GSDB year range:", gsdb["year"].min(), "-", gsdb["year"].max())

GSDB shape: (150200, 17)
GSDB year range: 1950 - 2023
CPU times: user 983 ms, sys: 307 ms, total: 1.29 s
Wall time: 1.51 s


In [ ]:
# MAIN direction: sender -> exporter, target -> importer
gsdb["pair"] = gsdb["sanctioning_state_iso3"] + "_" + gsdb["sanctioned_state_iso3"]

# ever sanctioned before the panel opens
pre_sanctioned = set(gsdb.loc[gsdb["year"] < PANEL_START, "pair"].unique())

# the binding case: a spell still RUNNING in 1994, i.e. it crosses into the panel
running_at_entry = set(gsdb.loc[gsdb["year"] == PANEL_START - 1, "pair"].unique())

print(f"dyads sanctioned at some point before {PANEL_START}: {len(pre_sanctioned):,}")
print(f"dyads with a spell running in {PANEL_START - 1}:        {len(running_at_entry):,}")

df["pre_sample_sanctioned"] = df["pair"].isin(running_at_entry).astype(int)

dyads sanctioned at some point before 1995: 5,103
dyads with a spell running in 1994:        3,659


In [ ]:
# MAIN direction: sender -> exporter, target -> importer
gsdb["pair"] = gsdb["sanctioning_state_iso3"] + "_" + gsdb["sanctioned_state_iso3"]

# ever sanctioned before the panel opens
pre_sanctioned = set(gsdb.loc[gsdb["year"] < PANEL_START, "pair"].unique())

# the binding case: a spell still RUNNING in 1994, i.e. it crosses into the panel
running_at_entry = set(gsdb.loc[gsdb["year"] == PANEL_START - 1, "pair"].unique())

print(f"dyads sanctioned at some point before {PANEL_START}: {len(pre_sanctioned):,}")
print(f"dyads with a spell running in {PANEL_START - 1}:        {len(running_at_entry):,}")

df["pre_sample_sanctioned"] = df["pair"].isin(running_at_entry).astype(int)

# --- generalise the pre-sample check to LATE ENTRANTS ---
# The 1994 check assumes every dyad enters in 1995. Late entrants (SSD 2011,
# ex-YUG/SUN successor states) enter later; "running in 1994" both misses their
# true pre-entry spells AND wrongly attributes predecessor-state sanctions to
# them. Tie each dyad's pre-history to the year before ITS OWN entry instead.
entry_year = df.groupby("pair")["year"].min()          # first observed year per dyad
gsdb_on = set(zip(gsdb["pair"], gsdb["year"]))          # (pair, year) sanctioned

pre_entry_running = {
    p: ((p, int(entry_year[p]) - 1) in gsdb_on)
    for p in entry_year.index
}
pre_entry = pd.Series(pre_entry_running).astype(int).rename("pre_entry_sanctioned")

print(f"left-censored (entry-year check): {int(pre_entry.sum()):,}")

dyads sanctioned at some point before 1995: 5,103
dyads with a spell running in 1994:        3,659
left-censored (entry-year check): 3,486


Scope: sanctions imposed 1995–2020. The panel starts in 1995 because the outcome does — BACI trade (trade), the key variable, has no coverage before then. Sanctions run back to 1950 in GSDB, so the two series overlap only from 1995. This cell measures what that costs: 3,659 dyads were already under sanction in 1994. They can't anchor an onset event study (no pre-period in the window), so they're excluded from onset and kept for termination, where the spell's end is still observed.

This is a scope condition, not a flaw: the project estimates the effect of sanctions imposed 1995–2020. The long comprehensive programmes (Cuba, DPRK, Iran) began earlier and sit in the excluded group, so onset results speak to the more recent, lighter, more targeted sanctions. The 1995 floor also keeps the panel identical to the companion ML project.

## (c) Spells, onsets, terminations

`contiguous` guards against treating a panel-entry gap as a status change.

In [ ]:
df = df.sort_values(["pair", "year"]).reset_index(drop=True)

g = df.groupby("pair", sort=False)
df["sanc_lag"]   = g["sanctioned_any"].shift(1)
df["year_lag"]   = g["year"].shift(1)
df["contiguous"] = (df["year"] - df["year_lag"] == 1)

# ONSET: 0 -> 1 across consecutive OBSERVED years
df["onset"] = ((df["sanctioned_any"] == 1) &
               (df["sanc_lag"] == 0) &
               (df["contiguous"])).astype(int)

# TERMINATION: 1 -> 0 across consecutive OBSERVED years
df["termination"] = ((df["sanctioned_any"] == 0) &
                     (df["sanc_lag"] == 1) &
                     (df["contiguous"])).astype(int)

print(f"onset events:       {int(df['onset'].sum()):,}")
print(f"termination events: {int(df['termination'].sum()):,}")

onset events:       6,613
termination events: 5,369


In [ ]:
# spell index within dyad: increments at every status change
df["status_change"] = df["onset"] + df["termination"]
df["spell_id"] = df.groupby("pair", sort=False)["status_change"].cumsum()

# number of distinct ON-spells per dyad
on_spells = (df[df["sanctioned_any"] == 1]
             .groupby("pair")["spell_id"].nunique()
             .rename("n_on_spells"))

print(on_spells.value_counts().sort_index().to_string())

n_on_spells
1    5988
2    1514
3     108
4      16


#### Dyad-level summary

`G_onset` is the cohort variable for staggered estimators: the year of the **first in-sample onset**. Left-censored dyads get `NaN` and are excluded from onset analysis.

In [ ]:
first_onset = df[df["onset"] == 1].groupby("pair")["year"].min().rename("G_onset")
first_term  = df[df["termination"] == 1].groupby("pair")["year"].min().rename("G_term")
ever_sanc   = df.groupby("pair")["sanctioned_any"].max().rename("ever_sanctioned")

first_year_status = (df.sort_values(["pair", "year"])
                       .groupby("pair")
                       .first()[["year", "sanctioned_any", "pre_sample_sanctioned"]]
                       .rename(columns={"year": "first_obs_year",
                                        "sanctioned_any": "status_at_entry"}))

dyad = (first_year_status
        .join(ever_sanc)
        .join(on_spells)
        .join(first_onset)
        .join(first_term)
        .join(pre_entry))                                    # <-- added
dyad["n_on_spells"] = dyad["n_on_spells"].fillna(0).astype(int)
dyad["pre_entry_sanctioned"] = dyad["pre_entry_sanctioned"].fillna(0).astype(int)  # <-- added

print("dyad table:", dyad.shape)
dyad.sample(8)

dyad table: (55922, 8)


,first_obs_year,status_at_entry,pre_sample_sanctioned,ever_sanctioned,n_on_spells,G_onset,G_term,pre_entry_sanctioned
pair,,,,,,,,
JOR_BEL,1995,0,0,0,0,NaN,NaN,0
LTU_AZE,1995,1,1,1,1,NaN,NaN,1
DOM_NER,1995,0,0,0,0,NaN,NaN,0
MOZ_SSD,2011,0,0,1,1,2015.0,NaN,0
PER_SGP,1995,0,0,0,0,NaN,NaN,0
MYT_SOM,1995,0,0,0,0,NaN,NaN,0
PRK_DJI,1995,0,0,0,0,NaN,NaN,0
TCA_TCD,1995,0,0,0,0,NaN,NaN,0


## (d) Classify dyads into estimation groups

| Group | Meaning | Onset sample | Termination sample |
|---|---|---|---|
| `never_treated` | no sanction ever, in or before panel | control | — |
| `single_onset` | enters untreated, one on-spell | treated | if it ends |
| `multi_spell` | enters untreated, switches on more than once | treated | if it ends |
| `left_censored_confirmed` | spell running in 1994 | **excluded** | included if it ends |
| `left_censored_ambiguous` | sanctioned at entry, no spell confirmed the year before entry | **excluded** | included if it ends |
| `treated_before_only` | pre-1995 spell ended before the panel opens | control | — |

Left-censoring is defined relative to each dyad's own panel entry, not a fixed 1994 baseline. This prevents predecessor-state sanction history (Yugoslavia -> Montenegro/Serbia, USSR successors) from being wrongly attributed to successor states that did not exist at the time, and correctly catches late entrants (e.g. South Sudan, 2011) sanctioned before their first appearance in the panel.


In [ ]:
def classify(r):
    # never sanctioned in-sample and no pre-sample spell
    if r["ever_sanctioned"] == 0 and r["pre_entry_sanctioned"] == 0:      # <-- changed
        return "never_treated"
    # sanctioned in its first observed year -> onset is unobservable
    if r["status_at_entry"] == 1:
        return ("left_censored_confirmed" if r["pre_entry_sanctioned"] == 1   # <-- changed
                else "left_censored_ambiguous")
    # enters untreated and switches on
    if r["n_on_spells"] == 1:
        return "single_onset"
    if r["n_on_spells"] > 1:
        return "multi_spell"
    # untreated in-sample, any pre-entry spell already over
    return "treated_before_only"

dyad["group"] = dyad.apply(classify, axis=1)
print(dyad["group"].value_counts().to_string())

group
never_treated              47600
single_onset                4065
left_censored_confirmed     2628
multi_spell                  770
treated_before_only          696
left_censored_ambiguous      163


In [ ]:
# look at ambiguous ones - possible mismatch between datasets?
amb = dyad[dyad["group"] == "left_censored_ambiguous"].index
first_sanc_year = (df[df["pair"].isin(amb) & (df["sanctioned_any"] == 1)]
                   .groupby("pair")["year"].min())
print(first_sanc_year.value_counts().sort_index().head(15))

year
1995    86
2002     9
2006    15
2011    53
Name: count, dtype: int64


In [ ]:
amb = dyad[dyad["group"] == "left_censored_ambiguous"].index
first_sanc = (df[df["pair"].isin(amb) & (df["sanctioned_any"] == 1)]
              .groupby("pair")["year"].min())
entry = dyad.loc[amb, "first_obs_year"]

mismatch = (first_sanc != entry).sum()
print(f"ambiguous dyads whose first sanction != entry year: {mismatch}")

ambiguous dyads whose first sanction != entry year: 0


No, they seem legit- just some countries are sanctioned the moment they start existing (inheritance artifacts and genuinly sanctioned). But all of them are excluded anyway - as they have no pre-onset baseline.

In [ ]:
# --- estimation flags ---

# Onset sample: clean switchers + valid controls. Left-censored dyads excluded:
# they carry a level difference, not an event.
dyad["in_onset_sample"] = dyad["group"].isin(
    ["single_onset", "multi_spell", "never_treated", "treated_before_only"]
).astype(int)

# Termination sample: any dyad whose spell ENDS in-window, including the
# left-censored ones -- their spell end is observable even if the onset is not.
dyad["in_term_sample"] = dyad["G_term"].notna().astype(int)

n_switch  = int(dyad["group"].isin(["single_onset", "multi_spell"]).sum())
n_lc_term = int(((dyad["in_term_sample"] == 1) &
                 dyad["group"].str.startswith("left_censored")).sum())

print(f"dyads usable for ONSET analysis:       {int(dyad['in_onset_sample'].sum()):,}")
print(f"  of which actual switchers:           {n_switch:,}")
print(f"dyads usable for TERMINATION analysis: {int(dyad['in_term_sample'].sum()):,}")
print(f"  of which left-censored:              {n_lc_term:,}")

dyads usable for ONSET analysis:       53,131
  of which actual switchers:           4,835
dyads usable for TERMINATION analysis: 4,277
  of which left-censored:              1,740


## (e) Transitions table

The feasibility gate. Three numbers decide the shape of the project:

1. **actual switchers** — under ~200 and Callaway–Sant'Anna group-time ATTs are too noisy to plot
2. **termination events** — under ~100 and terminations become descriptive rather than estimated
3. **zero-trade shares pre- vs post-onset** — if identical, the extensive margin is a null

In [ ]:
# how much staggering is there really?
cohort_sizes = (dyad.loc[dyad["G_onset"].notna(), "G_onset"]
                .value_counts().sort_index().rename("n_dyads"))
print("Onset cohorts (dyads by first-onset year):")
print(cohort_sizes.to_string())

thin = int((cohort_sizes < 5).sum())
print(f"\ncohorts with fewer than 5 dyads: {thin} of {len(cohort_sizes)}")
if len(cohort_sizes) and thin > 0.5 * len(cohort_sizes):
    print("  WARNING: most cohorts are thin. Group-time ATTs will be very noisy; "
          "plan to aggregate cohorts or rely on ETWFE-Poisson.")

Onset cohorts (dyads by first-onset year):
G_onset
1996.0    299
1997.0    233
1998.0    771
1999.0    112
2000.0    431
2001.0     54
2002.0    210
2003.0    275
2004.0    276
2005.0    503
2006.0    230
2007.0     39
2008.0     29
2009.0    215
2010.0     67
2011.0    273
2012.0    476
2013.0    183
2014.0    570
2015.0    201
2016.0      2
2017.0    177
2018.0      8
2019.0     33
2020.0     36

cohorts with fewer than 5 dyads: 1 of 25


Reading cohort table: "1998.0 -> 771" means 771 directed dyads had their first-ever sanction onset in 1998.

 Note: USA->Iran and Iran->USA are two different dyads.

Only 2016 is thin, not really a problem.

In [ ]:
# within-dyad transitions by sanction instrument.
# This is what identifies the disaggregated 'which instrument' question --
# and what the ML project's single-variable LASSO selection could not speak to.
sanc_types = [c for c in ["sanc_arms", "sanc_military", "sanc_trade",
                          "sanc_financial", "sanc_travel", "sanc_other",
                          "sender_mult", "target_mult"] if c in df.columns]

rows = []
for c in sanc_types:
    lag = df.groupby("pair", sort=False)[c].shift(1)
    rows.append({
        "variable":   c,
        "switch_on":  int(((df[c] == 1) & (lag == 0) & df["contiguous"]).sum()),
        "switch_off": int(((df[c] == 0) & (lag == 1) & df["contiguous"]).sum()),
        "dyads_ever": int(df.groupby("pair")[c].max().sum()),
    })

trans = pd.DataFrame(rows)
print(trans.to_string(index=False))

      variable  switch_on  switch_off  dyads_ever
     sanc_arms       3999        3426        5560
 sanc_military       3864        3493        4540
    sanc_trade       2564        3103        3609
sanc_financial       5096        3903        5953
   sanc_travel       4915        3390        5663
    sanc_other       1948        2483        2824
   sender_mult       6272        5126        7321
   target_mult        973        1494        1242


In [ ]:
# extensive margin: does trade actually go to zero around events?
never = dyad[dyad["group"] == "never_treated"].index
sw    = dyad[dyad["group"].isin(["single_onset", "multi_spell"])].index

d_never = df[df["pair"].isin(never)]
d_sw    = df[df["pair"].isin(sw)]

print("Zero-trade shares")
print(f"  never-treated dyads:    {d_never['trade'].eq(0).mean():.3f}")
print(f"  switchers, pre-onset:   {d_sw.loc[d_sw['sanctioned_any'] == 0, 'trade'].eq(0).mean():.3f}")
print(f"  switchers, post-onset:  {d_sw.loc[d_sw['sanctioned_any'] == 1, 'trade'].eq(0).mean():.3f}")

Zero-trade shares
  never-treated dyads:    0.538
  switchers, pre-onset:   0.369
  switchers, post-onset:  0.328


Zero-trade shares — motivation only, not a result. Roughly a third of switcher dyad-years show zero trade (pre-onset 0.369, post-onset 0.328), against 0.538 for never-treated dyads. The switcher–vs–never-treated gap is selection: countries are sanctioned where a trade relationship exists worth targeting, so sanctioned dyads start out more trade-active. The raw pre/post difference is likewise composition, not effect — it pools different dyads across the two buckets with no controls. These numbers establish only that the extensive margin is large enough to be worth modelling.

## Export

Merge the timing variables back onto the panel and save. `rel_onset` / `rel_term` are relative event time; `trade_pos` is the extensive-margin outcome.

In [ ]:
keep = ["group", "G_onset", "G_term", "in_onset_sample", "in_term_sample",
        "n_on_spells", "first_obs_year"]
df = df.merge(dyad[keep], left_on="pair", right_index=True, how="left")

# relative event time (NaN for never-treated -- estimators handle this)
df["rel_onset"] = np.where(df["G_onset"].notna(), df["year"] - df["G_onset"], np.nan)
df["rel_term"]  = np.where(df["G_term"].notna(),  df["year"] - df["G_term"],  np.nan)

# extensive-margin outcome
df["trade_pos"] = (df["trade"] > 0).astype(int)

df.drop(columns=["sanc_lag", "year_lag", "status_change"], inplace=True)

print("final panel:", df.shape)
df[["pair", "year", "trade", "sanctioned_any", "group",
    "G_onset", "rel_onset", "trade_pos"]].sample(5)

final panel: (1404170, 70)


,pair,year,trade,sanctioned_any,group,G_onset,rel_onset,trade_pos
551492,HKG_SYR,2010,16969.826,0,never_treated,NaN,NaN,1
396151,EST_BHR,2010,474.462,0,never_treated,NaN,NaN,1
884081,MWI_PHL,1995,0.000,0,never_treated,NaN,NaN,0
801926,MEX_NIU,2005,0.000,0,never_treated,NaN,NaN,0
152237,BLR_NAM,2001,0.000,0,never_treated,NaN,NaN,0


In [ ]:
%%time
df.to_csv(OUT_PATH + "panel_timing.csv", index=False)
dyad.to_csv(OUT_PATH + "dyad_classification.csv")
trans.to_csv(OUT_PATH + "transitions.csv", index=False)

print("saved ->", OUT_PATH + "panel_timing.csv", df.shape)
print("saved ->", OUT_PATH + "dyad_classification.csv", dyad.shape)
print("saved ->", OUT_PATH + "transitions.csv", trans.shape)

saved -> /content/drive/MyDrive/IMT_studies/Applied_econ/Project/Data/panel_timing.csv (1404170, 70)
saved -> /content/drive/MyDrive/IMT_studies/Applied_econ/Project/Data/dyad_classification.csv (55922, 11)
saved -> /content/drive/MyDrive/IMT_studies/Applied_econ/Project/Data/transitions.csv (8, 4)
CPU times: user 1min 7s, sys: 732 ms, total: 1min 8s
Wall time: 1min 12s
